# Daily Challenge: How to Finetune LLMs with LoRA

This notebook fine-tunes a pretrained language model with LoRA using the Hugging Face PEFT library. The goal is to adapt `bigscience/bloomz-560m` on a sample of the `Abirate/english_quotes` dataset, save the adapter, reload it, and generate text with the fine-tuned LoRA model.

## 1. Install Libraries

PEFT lets us fine-tune only a small number of adapter parameters instead of updating the full language model.

In [ ]:
%pip install -q peft==0.4.0 datasets transformers accelerate

In [ ]:
import os
import time

import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

os.makedirs("cache", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2. Load the Foundation Model and Tokenizer

`bigscience/bloomz-560m` is a causal language model. Since it predicts the next token, we use `task_type="CAUSAL_LM"` in the LoRA configuration later.

In [ ]:
model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
foundation_model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded:", model_name)
print("Tokenizer vocab size:", tokenizer.vocab_size)

## 3. Load and Preprocess the Quotes Dataset

The challenge asks for a 10% sample of the training split. We tokenize the `quote` field and keep a small maximum sequence length so the notebook remains practical on CPU or Colab.

In [ ]:
raw_data = load_dataset("Abirate/english_quotes", split="train[:10%]")
print(raw_data)
raw_data[0]

In [ ]:
MAX_LENGTH = 96

def tokenize_quotes(samples):
    tokenized = tokenizer(
        samples["quote"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(tokenize_quotes, batched=True, remove_columns=raw_data.column_names)

# Keep training short for the challenge notebook. Increase this for better adaptation.
train_dataset = data.select(range(min(128, len(data))))
train_sample = train_dataset.select(range(min(5, len(train_dataset))))
train_sample

In [ ]:
for i in range(len(train_sample)):
    print(f"Sample {i + 1}:")
    print(tokenizer.decode(train_sample[i]["input_ids"], skip_special_tokens=True))
    print("-" * 80)

## 4. Configure LoRA

For BLOOM/BLOOMZ models, common target modules include the fused query-key-value projection `query_key_value`. We use a small rank (`r=1`) as requested in the hint, which keeps the number of trainable parameters very low.

In [ ]:
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

## 5. Train the LoRA Adapter

The training configuration uses a higher learning rate than full fine-tuning because only a small adapter is being trained. The challenge hint uses CPU; the code below automatically uses CPU or GPU depending on the environment.

In [ ]:
output_directory = os.path.join("cache", "working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    use_cpu=(DEVICE == "cpu"),
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

## 6. Save the Fine-Tuned LoRA Model

Only the LoRA adapter weights are saved here, not a full copy of the foundation model. This is one of the main advantages of PEFT.

In [ ]:
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

print("Saved LoRA adapter to:", peft_model_path)
print("Saved files:", os.listdir(peft_model_path))

## 7. Load the Saved LoRA Model for Inference

We reload a fresh foundation model, then attach the saved LoRA adapter with `PeftModel.from_pretrained`. `is_trainable=False` means the adapter is loaded for inference only.

In [ ]:
base_model_for_inference = AutoModelForCausalLM.from_pretrained(model_name)
base_model_for_inference.config.pad_token_id = tokenizer.pad_token_id

loaded_peft_model = PeftModel.from_pretrained(
    base_model_for_inference,
    peft_model_path,
    is_trainable=False,
)

loaded_peft_model.to(DEVICE)
loaded_peft_model.eval()

print("LoRA adapter loaded for inference.")

## 8. Generate Text

The prompt is quote-like, matching the dataset style. Generation settings can be adjusted: lower temperature is more conservative, higher temperature is more creative.

In [ ]:
prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## 9. Quick Comparison: Base Model vs LoRA Model

This optional comparison uses the same prompt for the original foundation model and the LoRA-adapted model. With only a tiny training sample and one epoch, the difference may be small, but the workflow demonstrates the full PEFT process.

In [ ]:
foundation_model.to(DEVICE)
foundation_model.eval()

with torch.no_grad():
    base_outputs = foundation_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Base model output:")
print(tokenizer.batch_decode(base_outputs, skip_special_tokens=True)[0])
print("\nLoRA fine-tuned output:")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## Final Reflection

LoRA is parameter-efficient because it keeps the pretrained model weights frozen and trains small low-rank adapter matrices. This reduces memory usage, training time, and storage cost. Instead of saving hundreds of millions of model parameters, we save only the adapter files.

In this challenge, the adapter is trained on English quotes, so the intended behavior is to nudge the model toward quote-style completions. For a stronger result, increase the number of training samples, train for more epochs, and run on a GPU. The workflow stays the same: load the base model, configure LoRA, train with `Trainer`, save the adapter, reload it with `PeftModel.from_pretrained`, and generate text.